# A daily dependency watch with LangGraph and Web Search on Amazon Bedrock AgentCore

A security team that ships software has to know, every morning, whether anything published
since yesterday affects the open source packages and base images it depends on: a new CVE,
an advisory, a malicious package in the same ecosystem, or a vulnerability added to a
known-exploited list. Reading that by hand doesn't scale, and an agent that answers from
stale or uncited sources is worse than no agent.

This notebook builds that watch as a LangGraph graph on top of
[Web Search on Amazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html):

![Architecture of the dependency watch](images/agentcore_web_search_supply_chain_watch.svg)

What each part demonstrates, checked against live results in this notebook:

1. **Fresh results only, enforced in code.** Every search carries a published-after bound
   set by your code from the watch window. The model never sees a date argument, so it
   can't widen it. The bound applies to each article's publication date, not to the
   incident it describes, so the branch prompt also tells the model to report only issues
   first disclosed inside the window.
2. **A source blocklist the agent can't lift.** The gateway target carries an
   administrator blocklist applied server-side to every search. The companion notebook
   `agentcore_web_search_langgraph_walkthrough.ipynb` demonstrates it against live results.
3. **Parallel branches.** One branch per ecosystem, run concurrently by LangGraph, each
   with its own search client.
4. **Every finding backed by its source, and checked in code.** Branches return structured
   findings with a source URL. Code keeps a finding only if that URL came back from the
   branch's own searches, the returned text names the package, and any CVE or advisory
   ID the finding gives appears in that text.
5. **Only what's new.** A LangGraph store remembers what was already reported, so a second
   run the same day reports nothing twice.

Queries are served inside AWS and aren't sent to a third-party search engine, which
matters here: a dependency watch puts your package names and versions into its queries.

## Prerequisites

- An AWS account with Amazon Bedrock AgentCore access, working in a Region that offers the
  Web Search Tool (see [Supported AWS Regions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html)). In other Regions,
  creating the target fails with "Connector integration web-search is not available for this
  account", the same message you get when the connector isn't enabled for the account, so
  check the Region first.
- AWS credentials configured as environment variables (`AWS_ACCESS_KEY_ID`,
  `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`) or through an AWS CLI profile.
- Python 3.10 or later.
- Model access in Amazon Bedrock for the Claude model used below. Any Bedrock chat model that
  supports tool use works.
- IAM permissions for your identity to create, read and delete AgentCore gateways and gateway
  targets, `bedrock-agentcore:InvokeGateway` on the gateway, and `bedrock:InvokeModel`. If you let
  the notebook create the gateway's service role in Step 1, you also need `iam:GetRole`,
  `iam:CreateRole`, `iam:PutRolePolicy` and `iam:PassRole`.

## Cost and terms

Web Search and AgentCore Gateway are billed per use; see [Amazon Bedrock AgentCore pricing](https://aws.amazon.com/bedrock/agentcore/pricing/). One run of this notebook makes a few dozen searches plus the model calls. The
Web Search terms require that the source citations and links that come with each result
are kept and shown in any output that reaches an end user, and the report below does that
for every row.

## Install the packages and configure the watch

In [ ]:
%pip install -q -U langgraph langchain "langchain-aws[tools]>=1.8.0"

In [1]:
import json
import operator
import os
import re
import time
from datetime import datetime, timedelta, timezone
from typing import Annotated, Literal, Optional, TypedDict
from urllib.parse import urlparse

from bedrock_agentcore.gateway.client import GatewayClient
from bedrock_agentcore.tools import WebSearchClient
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langgraph.types import Send
from pydantic import BaseModel, Field

# Set the Region explicitly rather than reading AWS_REGION, and pick one that offers the
# Web Search Tool: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html
# An unsupported ambient Region creates a gateway and then fails the target with a
# message that reads like an entitlement problem.
REGION = "us-east-1"

MODEL_ID = "bedrock_converse:us.anthropic.claude-sonnet-5"
model = init_chat_model(MODEL_ID, region_name=REGION)

# What this team ships on. Replace with your own dependency inventory.
WATCHLIST = {
    "npm": ["express", "axios", "lodash"],
    "PyPI": ["requests", "urllib3", "cryptography"],
    "container base images": ["openssl", "glibc", "zlib"],
}

# Only material published inside this look-back window can reach the report. The store in
# Step 4 stops anything being reported twice, so a daily run with a week of look-back
# still reports only what's new.
WINDOW_DAYS = 7
MAX_SEARCHES_PER_BRANCH = 4


def domain_of(url):
    host = urlparse(url).hostname or ""
    return host[4:] if host.startswith("www.") else host


def matches(url, domains):
    host = domain_of(url)
    return any(host == d or host.endswith("." + d) for d in domains)


def normalize(url):
    return url.rstrip("/").split("#")[0]


def parse_published(value):
    # The index reports dates like "05:00PM, Tuesday, October 06 2026, PDT".
    if not value:
        return None
    stamp, _, zone = value.rpartition(", ")
    offsets = {"PDT": -7, "PST": -8, "UTC": 0, "GMT": 0}
    try:
        local = datetime.strptime(stamp, "%I:%M%p, %A, %B %d %Y")
    except ValueError:
        return None
    return (local - timedelta(hours=offsets.get(zone, 0))).replace(tzinfo=timezone.utc)

## Step 1: Create the gateway and a web search target with a blocklist

Web search reaches the service as an AgentCore Gateway connector target. The target here,
`security-sources`, carries an **administrator blocklist**: domains your security team has
decided not to use as evidence. It's applied server-side to every search through the
target and is hidden from the agent, which can't lift it. The default below blocks AWS's
own marketing site, so the policy is visible in the results without naming anyone else's;
put your team's real list here.

The target is pinned to connector version `1.2.0`, the first version with request-level
filters, which the date window relies on.

Two IAM grants, on two different principals:

- The **gateway's service role** needs
  `bedrock-agentcore:InvokeWebSearch` on
  `arn:aws:bedrock-agentcore:<region>:aws:tool/web-search.v1`, and a trust policy that lets
  `bedrock-agentcore.amazonaws.com` assume it. If the trust policy scopes `aws:SourceArn`,
  the region in it must be the gateway's region, or the first search fails with
  `Failed to obtain execution role credentials`.
- **Your own credentials**, which the search client signs with, need
  `bedrock-agentcore:InvokeGateway` on the gateway.

The first cell creates the service role if it doesn't exist yet, or uses the role in
`GATEWAY_ROLE_ARN` if you set one. The second cell creates the gateway and target, reuses them
if they already exist, and fails loudly if a reused target is on an older connector version.

In [2]:
import boto3

GATEWAY_ROLE_NAME = "AgentCoreWebSearchGatewayRole"
iam = boto3.client("iam")
account_id = boto3.client("sts").get_caller_identity()["Account"]

if os.environ.get("GATEWAY_ROLE_ARN"):
    GATEWAY_ROLE_ARN = os.environ["GATEWAY_ROLE_ARN"]
else:
    try:
        GATEWAY_ROLE_ARN = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        trust = {
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"},
                },
            }],
        }
        GATEWAY_ROLE_ARN = iam.create_role(
            RoleName=GATEWAY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust),
            Description="Lets AgentCore Gateway call AgentCore Web Search",
        )["Role"]["Arn"]
        iam.put_role_policy(
            RoleName=GATEWAY_ROLE_NAME, PolicyName="InvokeWebSearch",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": "bedrock-agentcore:InvokeWebSearch",
                    "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:tool/web-search.v1",
                }],
            }),
        )
        time.sleep(10)  # let the new role propagate before the gateway assumes it
print(f"gateway service role: {GATEWAY_ROLE_ARN.split('/')[-1]}")

gateway service role: AgentCoreWebSearchGatewayRole


In [3]:
GATEWAY_NAME = "langchain-supply-chain-watch"
TARGET_NAME = "security-sources"
ADMIN_BLOCKLIST = ["amazon.com"]
CONNECTOR_VERSION = "1.2.0"

gateway_client = GatewayClient(region_name=REGION)
gateway = gateway_client.get_gateway_by_name(GATEWAY_NAME)
if gateway is None:
    gateway = gateway_client.create_gateway_and_wait(
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="AWS_IAM",
        protocolType="MCP",
    )
GATEWAY_ID = gateway["gatewayId"]

target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME)
if target is None:
    target = gateway_client.create_web_search_target(
        gateway_identifier=GATEWAY_ID,
        name=TARGET_NAME,
        exclude_domains=ADMIN_BLOCKLIST,
        connector_version=CONNECTOR_VERSION,
    )
connector = target["targetConfiguration"]["mcp"]["connector"]
if connector["source"].get("version") != CONNECTOR_VERSION:
    raise RuntimeError(f"{TARGET_NAME} is on connector {connector['source'].get('version')}")
print(f"{TARGET_NAME}: {target['status']}, connector {CONNECTOR_VERSION}, "
      f"admin policy {connector['configurations'][0]['parameterValues']}")

security-sources: READY, connector 1.2.0, admin policy {'domainFilter': {'exclude': ['amazon.com']}}


## Step 2: Create a search tool with a fixed date window

Each research branch gets its own tool built on the AgentCore SDK's `WebSearchClient`, the client that
`create_web_search_toolkit` wraps. The toolkit's `web_search` tool lets the model choose its filters;
here the date window has to come from code, so the tool wraps the client directly. The
tool takes only a query from the model. The published-after bound comes from the run's
watch window and is added to every call, so the model can't search older material even if
it tries. The tool returns the results twice: as text for the model, and as structured data
(an artifact) that the graph uses later to check each finding against its source. The tool
also caps the number of searches per branch in code, so the cost of a run is bounded no
matter what the model does.

Each branch creates its **own** client, so the branches' searches run concurrently.

In [4]:
def make_search_tool(window_start, max_searches=MAX_SEARCHES_PER_BRANCH):
    # One client per tool: calls on a single client run one at a time.
    client = WebSearchClient(region=REGION, gateway_id=GATEWAY_ID, target_name=TARGET_NAME)
    published_after = window_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    calls = {"count": 0}

    @tool(response_format="content_and_artifact")
    def search_security_news(query: str) -> tuple:
        '''Search recent security advisories, CVE records and security news. Cite the URLs you use.'''
        calls["count"] += 1
        if calls["count"] > max_searches:
            return f"Search limit of {max_searches} reached. Report what you have.", []
        response = client.search(query, max_results=8, published_after=published_after)
        sources = [
            {"title": r.title, "url": r.url, "published": r.published_date, "text": r.text}
            for r in response.results if r.url
        ]
        text = "\n\n".join(
            f"{i}. {r.title or r.url}\n   URL: {r.url}\n   Published: {r.published_date}\n   {r.text[:700]}"
            for i, r in enumerate(response.results, start=1) if r.url
        ) or "No results in the watch window."
        return text, sources

    search_security_news.handle_tool_error = True
    return search_security_news, client


window_start = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
probe_tool, probe_client = make_search_tool(window_start)
print("arguments the model sees:", list(probe_tool.tool_call_schema.model_json_schema()["properties"]))
# Calling the underlying function returns both parts: the text for the model and the artifact.
text, sources = probe_tool.func("known exploited vulnerabilities added")
print(f"\n{len(sources)} results for a window starting {window_start:%Y-%m-%d %H:%M} UTC:")
for s in sources[:5]:
    print(f"   {s['published']} | {domain_of(s['url'])}")
probe_client.close()

arguments the model sees: ['query']



8 results for a window starting 2026-10-07 14:50 UTC:
   05:00PM, Monday, October 12 2026, PDT | cisa.gov
   05:00AM, Tuesday, October 13 2026, PDT | cisa.gov
   09:34PM, Tuesday, October 13 2026, PDT | thehackernews.com
   05:00AM, Friday, October 09 2026, PDT | cisa.gov
   05:00PM, Tuesday, October 13 2026, PDT | threatlandscape.io


## Step 3: Define the findings schema and the branch prompt

Each branch is a small agent that searches, then returns a list of findings in a fixed
schema rather than prose. Every finding must name the package it affects and the URL of
the source it came from.

In [5]:
class Finding(BaseModel):
    package: str = Field(description="The watched package or image this affects, exactly as named in the watchlist.")
    title: str = Field(description="One line: what happened.")
    identifier: Optional[str] = Field(default=None, description="CVE or advisory ID if the source gives one.")
    severity: Literal["critical", "high", "medium", "low", "unknown"] = Field(
        default="unknown", description="The severity the source states. Use unknown if the source doesn't state one.")
    actively_exploited: bool = Field(description="True only if the source says it is exploited in the wild.")
    action: str = Field(description="What the team should do, in one line. Vendor-neutral: never recommend a commercial product or service.")
    source_url: str = Field(description="URL of the search result this finding came from. Must be one you were given.")
    published: Optional[str] = Field(default=None, description="Publication date the search result reported.")


class EcosystemFindings(BaseModel):
    findings: list[Finding] = Field(description="Empty if nothing in the window affects the watched packages.")


BRANCH_PROMPT = """You watch one software ecosystem for a security team.
Search for anything published in the watch window that affects the watched packages:
new CVEs, security advisories, malicious or compromised packages, and vulnerabilities
reported as exploited. Run at most four searches.

Report a finding only when a search result is about one of the watched packages itself:
a vulnerability, advisory or compromise in that package. A result that only mentions the
package in passing, or is about a different package that depends on it, is not a finding.
Use the exact source URL from the search results. Take severity only from what the source
states. Keep actions vendor-neutral, such as upgrading, patching or checking configuration,
and never recommend a commercial product or service. If nothing in the results affects the
watched packages, return no findings. Never fill gaps from memory.

Report only issues first disclosed inside the watch window. A recent result that describes
an older incident as background is not a finding."""

## Step 4: Build the graph

`fan_out` sends one branch per ecosystem with LangGraph's `Send`, so the branches run in
parallel. Each branch runs its agent and then **checks every finding against its source**:
the URL must have come back from that branch's searches, the returned text must name the
package as a whole word, any CVE or advisory ID must appear in that text, and the source's
publication date must fall inside the watch window. A missing or unparseable date counts as
outside. Findings that fail are dropped and counted by reason. `keep_new`
compares findings with a store of what was already reported, and `report` builds the
table.

In [6]:
class WatchState(TypedDict):
    watchlist: dict
    window_start: str
    findings: Annotated[list, operator.add]
    branch_stats: Annotated[list, operator.add]
    new_findings: list
    report: str


class BranchState(TypedDict):
    ecosystem: str
    packages: list
    window_start: str


def fan_out(state: WatchState):
    return [
        Send("research", {"ecosystem": eco, "packages": pkgs, "window_start": state["window_start"]})
        for eco, pkgs in state["watchlist"].items()
    ]


def research(branch: BranchState):
    started = time.perf_counter()
    window_start = datetime.fromisoformat(branch["window_start"])
    search_tool, client = make_search_tool(window_start)
    try:
        agent = create_agent(model, tools=[search_tool], system_prompt=BRANCH_PROMPT,
                             response_format=EcosystemFindings)
        result = agent.invoke({"messages": [{"role": "user", "content":
            f"Ecosystem: {branch['ecosystem']}. Watched packages: {', '.join(branch['packages'])}. "
            f"Watch window starts {window_start:%Y-%m-%d %H:%M} UTC."}]})
    finally:
        client.close()

    returned = {normalize(s["url"]): s
                for m in result["messages"] if m.type == "tool" for s in (m.artifact or [])}
    searches = sum(1 for m in result["messages"] if m.type == "tool" and m.artifact)
    kept, dropped = [], {"url_not_returned": 0, "package_not_in_source": 0, "id_not_in_source": 0,
                         "date_outside_window": 0}
    for finding in result["structured_response"].findings:
        source = returned.get(normalize(finding.source_url))
        evidence = ((source or {}).get("title") or "") + " " + ((source or {}).get("text") or "")
        published = parse_published(source["published"]) if source else None
        if source is None:
            dropped["url_not_returned"] += 1
        elif not re.search(rf"(?<![\w.-]){re.escape(finding.package)}(?![\w-])", evidence, re.IGNORECASE):
            dropped["package_not_in_source"] += 1
        elif finding.identifier and finding.identifier.lower() not in evidence.lower():
            dropped["id_not_in_source"] += 1
        elif published is None or published < window_start:
            # The search already asked for published-after; this makes it a hard rule.
            dropped["date_outside_window"] += 1
        else:
            kept.append({**finding.model_dump(), "ecosystem": branch["ecosystem"],
                         "source_published": source["published"]})
    return {
        "findings": kept,
        "branch_stats": [{
            "ecosystem": branch["ecosystem"], "searches": searches, "sources_returned": len(returned),
            "findings_kept": len(kept), "dropped": dropped,
            "blocked_domain_sources": sum(matches(u, ADMIN_BLOCKLIST) for u in returned),
            "seconds": round(time.perf_counter() - started, 1),
        }],
    }


def finding_key(f):
    return (f.get("identifier") or normalize(f["source_url"])).lower() + "|" + f["package"].lower()


def keep_new(state: WatchState, *, store):
    new = []
    for f in state["findings"]:
        key = finding_key(f)
        if store.get(("reported",), key) is None:
            store.put(("reported",), key, {"first_seen": datetime.now(timezone.utc).isoformat(), **f})
            new.append(f)
    return {"new_findings": new}


def report(state: WatchState):
    rows = sorted(state["new_findings"], key=lambda f: (not f["actively_exploited"],
                  ["critical", "high", "medium", "low", "unknown"].index(f["severity"])))
    if not rows:
        return {"report": "No new findings in the watch window."}
    lines = ["| Package | What happened | ID | Severity | Exploited | Action | Source |",
             "|---|---|---|---|---|---|---|"]
    for f in rows:
        lines.append(f"| {f['package']} ({f['ecosystem']}) | {f['title']} | {f.get('identifier') or ''} | "
                     f"{f['severity']} | {'yes' if f['actively_exploited'] else 'no'} | {f['action']} | "
                     f"[{domain_of(f['source_url'])}]({f['source_url']}) |")
    return {"report": "\n".join(lines)}


builder = StateGraph(WatchState)
builder.add_node("research", research)
builder.add_node("keep_new", keep_new)
builder.add_node("report", report)
builder.add_conditional_edges(START, fan_out, ["research"])
builder.add_edge("research", "keep_new")
builder.add_edge("keep_new", "report")
builder.add_edge("report", END)

# InMemoryStore keeps this notebook self-contained. For a scheduled job, use a durable
# store with exact key lookups, such as DynamoDBStore from langgraph-checkpoint-aws.
store = InMemoryStore()
watch = builder.compile(store=store)
print(watch.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	research(research)
	keep_new(keep_new)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> research;
	keep_new --> report;
	research --> keep_new;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 5: Run the watch

In [7]:
def run_watch():
    started = time.perf_counter()
    window_start = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
    state = watch.invoke({"watchlist": WATCHLIST, "window_start": window_start.isoformat(),
                          "findings": [], "branch_stats": []})
    return state, round(time.perf_counter() - started, 1)


state, wall = run_watch()
print(f"wall time {wall}s across {len(state['branch_stats'])} parallel branches "
      f"(sum of branch times {sum(b['seconds'] for b in state['branch_stats']):.1f}s)\n")
for b in state["branch_stats"]:
    print(b)
print(f"\nfindings kept: {len(state['findings'])}, new this run: {len(state['new_findings'])}")

wall time 22.0s across 3 parallel branches (sum of branch times 46.8s)

{'ecosystem': 'npm', 'searches': 4, 'sources_returned': 26, 'findings_kept': 3, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 15.8}
{'ecosystem': 'PyPI', 'searches': 4, 'sources_returned': 27, 'findings_kept': 1, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 9.0}
{'ecosystem': 'container base images', 'searches': 4, 'sources_returned': 21, 'findings_kept': 11, 'dropped': {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, 'blocked_domain_sources': 0, 'seconds': 22.0}

findings kept: 15, new this run: 15


In [8]:
from IPython.display import Markdown, display

display(Markdown(state["report"]))

| Package | What happened | ID | Severity | Exploited | Action | Source |
|---|---|---|---|---|---|---|
| axios (npm) | Compromised maintainer account used to publish malicious axios versions (1.14.1, 0.30.4) adding dependency 'plain-crypto-js' to deploy a cross-platform RAT |  | critical | yes | Immediately check installed axios versions, remove/upgrade away from 1.14.1 and 0.30.4, clear npm cache, remove the plain-crypto-js dependency if present, and review system logs for suspicious outbound connections. | [nca.gov.sa](https://nca.gov.sa/en/cert/7511/) |
| express (npm) | DirtyBlanket worm: typosquat npm packages impersonating Express (e.g. 'exprrdd') publish malicious Linux-targeting worm |  | high | yes | Verify exact package names before installing; audit CI/CD and developer machines for the dirtyblanket-published typosquat packages, rotate SSH keys/npm tokens if any were installed, and enable npm package name verification/allow-listing. | [baumannzone.dev](https://www.baumannzone.dev/blog/dirtyblanket-un-gusano-de-linux-escondido-en-falsos-paquetes-de-express) |
| openssl (container base images) | Use-after-free during PKCS#7/S-MIME signature verification (empty digestAlgorithms SET causes OpenSSL to incorrectly free a caller-owned BIO) | CVE-2026-45447 | high | no | Upgrade openssl to the patched version noted in the advisory (e.g. 1.1.1w-r10075 or later for Alpine 3.15-based images) and rebuild affected container images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-45447) |
| openssl (container base images) | DTLS retransmission flaw can disclose heap memory as plaintext handshake data or cause a crash/DoS | CVE-2026-84782 | high | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-84782) |
| openssl (container base images) | Signed integer overflow in ASN1_mbstring_ncopy() destination buffer sizing leads to heap buffer overflow | CVE-2026-7383 | high | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-7383) |
| express (npm) | Express v5.3.0 released fixing qs dependency vulnerabilities CVE-2026-2391 (GHSA-w7fw-mjwx-w883) and CVE-2026-82417 | CVE-2026-2391 | medium | no | Upgrade Express to 5.3.0 (or update the qs dependency to ^6.16.0) especially if parsing external query strings. | [tsumikasane.net](https://tsumikasane.net/releases/express/5.3.0) |
| openssl (container base images) | Integer truncation in ASN.1 decoder causes heap buffer over-read when parsing primitive elements over 2GB | CVE-2026-34180 | medium | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-34180) |
| openssl (container base images) | NULL pointer dereference in delta CRL processing when CRL Number extension is missing, leading to DoS | CVE-2026-28388 | medium | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-28388) |
| openssl (container base images) | NULL pointer dereference processing crafted CMS EnvelopedData with KeyTransportRecipientInfo, leading to DoS | CVE-2026-28390 | medium | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-28390) |
| openssl (container base images) | DTLS future-epoch record buffering allows memory amplification DoS | CVE-2026-54874 | medium | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-54874) |
| openssl (container base images) | Heap out-of-bounds read in kek_unwrap_key() during CMS password-based decryption (PWRI key unwrap) | CVE-2026-9076 | medium | no | Upgrade openssl to the patched version for your base image distribution and rebuild images. | [securityfocus.com](https://www.securityfocus.com/advisory/ROOT-OS-ALPINE-315-CVE-2026-9076) |
| cryptography (PyPI) | pyca/cryptography did not validate requested peer name against DNS Name Constraints in certificate SANs (RFC 5280 name-constraint inconsistency), fixed in 46.0.6 | CVE-2026-34073 | unknown | no | Upgrade cryptography to version 46.0.6 or later to ensure DNS Name Constraints are checked against the actually requested peer name. | [kb.cert.org](https://kb.cert.org/vuls/id/573195) |
| glibc (container base images) | SUSE security update addresses 10 vulnerabilities in glibc |  | unknown | no | Apply the SUSE glibc security update (SUSE-SU-2026:4591-1) or equivalent upstream patch to affected container base images. | [linuxsecurity.com](https://linuxsecurity.com/advisories/suse/suse-2026-4591-1-glibc) |
| glibc (container base images) | Process abort due to invalid memory in wordexp() | CVE-2026-6368 | unknown | no | Upgrade glibc to the fixed version provided by your distribution's security advisory and rebuild affected images. | [scout.docker.com](https://scout.docker.com/v/CVE-2026-6368?s=rocky) |
| glibc (container base images) | Denial of Service via stack exhaustion during tilde expansion | CVE-2026-6791 | unknown | no | Upgrade glibc to the fixed version provided by your distribution's security advisory and rebuild affected images. | [scout.docker.com](https://scout.docker.com/v/CVE-2026-6791?s=rocky) |

### Check the report against the date window and the blocklist

The branch statistics above count findings dropped by the evidence check, by reason. The
branches already drop anything outside the window; this cell re-checks the rows that survived,
independently of the graph, against the two rules that matter for an alert: the source was
published inside the window, and it isn't on the administrator blocklist.

In [9]:
run_window_start = datetime.fromisoformat(state["window_start"])
outside_window, undated, blocked = [], [], []
for f in state["findings"]:
    published = parse_published(f["source_published"])
    if published is None:
        undated.append(f["source_url"])
    elif published < run_window_start:
        outside_window.append((f["source_published"], f["source_url"]))
    if matches(f["source_url"], ADMIN_BLOCKLIST):
        blocked.append(f["source_url"])
print(f"findings: {len(state['findings'])}")
print(f"source published before the window: {len(outside_window)}")
print(f"source date not parseable: {len(undated)}")
print(f"from blocked domains: {len(blocked)}")
print(f"dropped across branches: {[b['dropped'] for b in state['branch_stats']]}")
print(f"blocked-domain sources returned by any search: {sum(b['blocked_domain_sources'] for b in state['branch_stats'])}")
for f in state["findings"][:5]:
    print(f"   {f['source_published']} | {f['package']} | {f['source_url']}")

findings: 15
source published before the window: 0
source date not parseable: 0
from blocked domains: 0
dropped across branches: [{'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}, {'url_not_returned': 0, 'package_not_in_source': 0, 'id_not_in_source': 0, 'date_outside_window': 0}]
blocked-domain sources returned by any search: 0
   05:00PM, Thursday, October 08 2026, PDT | express | https://www.baumannzone.dev/blog/dirtyblanket-un-gusano-de-linux-escondido-en-falsos-paquetes-de-express
   05:00PM, Friday, October 09 2026, PDT | express | https://tsumikasane.net/releases/express/5.3.0
   05:00PM, Friday, October 09 2026, PDT | axios | https://nca.gov.sa/en/cert/7511/
   05:00PM, Monday, October 12 2026, PDT | cryptography | https://kb.cert.org/vuls/id/573195
   09:15AM, Tuesday, October 13 2026, PDT | openssl | https://www.securityfocu

## Step 6: Run the watch again to see only new findings

A scheduled watch runs every day, and the team shouldn't see the same advisory twice. The
store from the first run already holds every reported finding, so a second run reports only
findings it hasn't seen.

In [10]:
state_2, wall_2 = run_watch()
print(f"second run: {len(state_2['findings'])} findings found, {len(state_2['new_findings'])} new")
print(state_2["report"] if state_2["new_findings"] else "nothing new to report")

second run: 2 findings found, 1 new
| Package | What happened | ID | Severity | Exploited | Action | Source |
|---|---|---|---|---|---|---|
| urllib3 (PyPI) | urllib3 streaming API vulnerability (CVE-2026-21441) flagged in Cisco third-party component bug tracker | CVE-2026-21441 | unknown | no | Review the CVE-2026-21441 details once a primary advisory is published and upgrade urllib3 to the patched release when available; in the meantime check whether your usage of urllib3's streaming/decode API is affected. | [quickview.cloudapps.cisco.com](https://quickview.cloudapps.cisco.com/quickview/bug/CSCws78525) |


## Next steps

- **Schedule it.** Run the graph once a day, or more often for a tight window, and post the
  report wherever the team works.
- **Use a durable store.** Swap `InMemoryStore` for a store with exact key lookups, such as
  `DynamoDBStore` from `langgraph-checkpoint-aws`, so "already reported" survives between runs.
- **Host it on AgentCore Runtime.** The `agentcore_web_search_supply_chain_runtime` folder next
  to this notebook has the same graph as an AgentCore Runtime entrypoint, with the steps to
  deploy it using the [AgentCore CLI](https://github.com/aws/agentcore-cli). Runtime session state
  is ephemeral, so pair it with the durable store above.
- **Tune the blocklist.** Put the sources your team doesn't accept as evidence on the target
  itself, so every agent using the gateway gets the same policy.

## Clean up

The gateway and target are left in place unless you set `DELETE_GATEWAY = True`. The IAM role is
left in place either way, so other gateways can use it; delete it in the IAM console when you no
longer need it.

In [11]:
DELETE_GATEWAY = False

if DELETE_GATEWAY:
    target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, TARGET_NAME)
    if target:
        gateway_client.delete_gateway_target_and_wait(gatewayIdentifier=GATEWAY_ID, targetId=target["targetId"])
    gateway_client.delete_gateway_and_wait(gatewayIdentifier=GATEWAY_ID)
    print(f"deleted {GATEWAY_ID}")
else:
    print("gateway left in place")

gateway left in place
